**Project UTS Computer Vision**


#**Sistem Presensi Mahasiswa Berbasis Pengenalan Wajah**

##**Link Youtube**

https://youtu.be/lMMAUuou9_Q?si=hfn192s1IgsO4ufD


##**Program Deteksi_Absensi**

###**Bagian 1 - Import Library**


1. cv2 → Merupakan library OpenCV yang digunakan untuk membuka kamera, membaca frame video, menggambar kotak pembatas pada wajah, serta menampilkan jendela kamera secara real-time.
2. face_recognition → Merupakan library utama yang menangani seluruh proses deteksi dan pengenalan wajah. Library ini memanfaatkan model deep learning dari dlib sebagai pondasi utamanya.
3. dlib → Merupakan library berbasis C++ yang menyediakan implementasi algoritma machine learning dan computer vision tingkat lanjut. Dalam konteks sistem ini, dlib berperan sebagai fondasi dari library face_recognition, di mana seluruh proses deteksi landmark wajah, ekstraksi fitur, serta komputasi encoding 128 dimensi dijalankan oleh model-model yang disediakan oleh dlib di balik layar. Library ini menggunakan metode Histogram of Oriented Gradients (HOG) untuk deteksi wajah serta Deep Residual Network untuk menghasilkan encoding wajah yang akurat.
4. os → Digunakan agar program dapat membaca nama file dalam sebuah folder, memeriksa keberadaan file, serta menggabungkan path folder dengan nama file secara dinamis.
5. pandas → Digunakan untuk membaca dan menulis file CSV sebagai media penyimpanan data absensi. Pandas dipilih karena mampu menangani data dalam bentuk tabel secara terstruktur, termasuk operasi filtering seperti pengecekan duplikasi data absensi pada hari yang sama.
6. datetime → Digunakan untuk mengambil informasi tanggal dan waktu pada saat seseorang berhasil terdeteksi oleh sistem.
7. numpy → Digunakan untuk operasi array numerik, khususnya fungsi np.argmin() yang bertugas mencari indeks nilai terkecil dari hasil perhitungan jarak antar encoding wajah.

Setiap library memiliki tanggung jawab yang berbeda dan saling melengkapi. Apabila pandas tidak digunakan, maka logika baca dan tulis file CSV harus ditulis secara manual menggunakan Python murni, yang berpotensi menimbulkan kesalahan logika. Apabila numpy tidak digunakan, proses pencarian wajah paling mirip dari sekumpulan data referensi tidak dapat dilakukan secara efisien.

Meskipun dlib tidak dipanggil secara langsung dalam kode program ini, keberadaannya tetap wajib karena seluruh fungsionalitas dari library face_recognition bergantung sepenuhnya padanya. Berikut adalah peran spesifik dlib dalam sistem ini:
1. Deteksi Wajah (Face Detection)
dlib menyediakan detektor wajah berbasis HOG + Linear SVM yang digunakan oleh face_recognition.face_locations() untuk menemukan posisi wajah pada sebuah gambar atau frame video. Algoritma HOG bekerja dengan menganalisis gradien intensitas piksel untuk menghasilkan deskriptor visual yang kemudian diklasifikasikan oleh model SVM.
2. Deteksi Landmark Wajah (Shape Predictor)
Setelah wajah ditemukan, dlib menggunakan model 68-point facial landmark predictor untuk mengidentifikasi titik-titik kunci pada wajah seperti sudut mata, ujung hidung, dan garis bibir. Informasi landmark ini digunakan untuk melakukan normalisasi posisi wajah sebelum encoding dilakukan.
3. Ekstraksi Encoding Wajah (Face Recognition Model)
dlib menyediakan model jaringan saraf tiruan yang telah dilatih sebelumnya (pre-trained) untuk mengonversi wajah yang telah dinormalisasi menjadi vektor encoding berisi 128 nilai numerik. Model ini dilatih menggunakan dataset berskala besar dan mampu menghasilkan encoding yang konsisten meskipun terdapat variasi pencahayaan, sudut pandang, dan ekspresi wajah.


dlib dipilih sebagai fondasi karena menawarkan akurasi pengenalan wajah yang tinggi dengan tingkat kesalahan yang rendah pada dataset pengujian standar. Model deep residual network yang disertakan dalam dlib telah mencapai akurasi di atas 99% pada benchmark Labeled Faces in the Wild (LFW), menjadikannya salah satu implementasi pengenalan wajah open-source yang paling andal hingga saat ini. Selain itu, dlib ditulis dalam bahasa C++ sehingga kecepatan komputasinya jauh lebih tinggi dibandingkan implementasi serupa yang ditulis dalam Python murni.

In [ ]:
import cv2
import face_recognition
import os
import pandas as pd
from datetime import datetime
import numpy as np

###**Bagian 2 — Pembacaan Foto Referensi dari Folder**


1. os.listdir() → Membaca seluruh nama file yang terdapat di dalam folder foto_wajah.
2. List comprehension dengan .endswith() → Berfungsi sebagai filter agar hanya file dengan ekstensi gambar (jpg/jpeg/png) yang diproses, sehingga file lain seperti .txt tidak menyebabkan kesalahan pada program.
3. os.path.join() → Menggabungkan nama folder dan nama file menjadi sebuah path yang valid. Penggunaan fungsi ini memastikan kompatibilitas program di berbagai sistem operasi, mengingat Windows menggunakan pemisah \ sementara Linux dan macOS menggunakan /.
4. cv2.imread() → Membaca file gambar dan mengonversinya menjadi array piksel dalam format BGR.
5. if img_bgr is not None → Pemeriksaan ini diperlukan karena apabila file gambar mengalami kerusakan atau formatnya tidak valid, fungsi cv2.imread() akan mengembalikan nilai None. Tanpa pengecekan ini, program akan terhenti secara tidak terduga.

Pendekatan ini memungkinkan sistem untuk membaca data referensi secara dinamis tanpa perlu menuliskan nama file secara manual di dalam kode. Penambahan foto baru ke dalam folder akan langsung dikenali oleh sistem tanpa memerlukan perubahan pada kode program.

In [ ]:
# Folder tempat foto referensi disimpan
path_gambar = 'foto_wajah'
images = []
daftar_file = os.listdir(path_gambar)

print("Memuat foto referensi dan mengekstrak fitur wajah...")

# Membaca setiap gambar di folder foto_wajah
gambar_files = [cl for cl in daftar_file if cl.lower().endswith(('.jpg', '.jpeg', '.png'))]
for cl in gambar_files:
    img_path = os.path.join(path_gambar, cl)
    img_bgr = cv2.imread(img_path)

    if img_bgr is not None:
        images.append(img_bgr)

###**Bagian 3 — Fungsi Ekstraksi Encoding Wajah**

1. cv2.cvtColor(..., cv2.COLOR_BGR2RGB) → OpenCV secara bawaan membaca gambar dalam format BGR, sedangkan library face_recognition memerlukan format RGB. Apabila konversi ini tidak dilakukan, urutan channel warna akan terbalik dan proses deteksi wajah tidak akan berjalan dengan benar.
2. face_recognition.face_encodings(img_rgb)[0] → Fungsi ini mengonversi wajah pada foto menjadi sebuah array numerik berisi 128 nilai yang merepresentasikan karakteristik unik wajah tersebut. Array ini berfungsi sebagai "sidik jari digital" setiap individu. Indeks [0] diambil dengan asumsi bahwa setiap foto referensi hanya mengandung satu wajah.
3. os.path.splitext(cl)[0].split('_')[0] → Logika ini mengekstrak nama individu dari nama file. Sebagai contoh, file bernama Budi_1.jpg akan menghasilkan nama Budi setelah diproses. Mekanisme ini memungkinkan penyimpanan beberapa foto referensi untuk satu individu yang sama.
4. try/except IndexError → Apabila sebuah foto tidak mengandung wajah yang dapat terdeteksi, fungsi face_encodings() akan mengembalikan list kosong dan pengaksesan indeks [0] akan memicu IndexError. Blok try/except ini mencegah program terhenti dan menggantinya dengan pesan peringatan yang informatif.


Model pengenalan wajah yang digunakan menghasilkan encoding 128 dimensi yang dirancang untuk bersifat unik per individu. Dua foto dari orang yang sama akan menghasilkan encoding dengan jarak yang kecil, sedangkan dua individu berbeda akan memiliki jarak encoding yang jauh. Inilah dasar matematis dari keseluruhan sistem pengenalan wajah ini.

In [ ]:
# Fungsi untuk mendapatkan encoding wajah dari gambar referensi
def findEncodings(images, gambar_files):
    encodeList = []
    classNames = []  # Buat lokal di sini
    for img, cl in zip(images, gambar_files):
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        try:
            encode = face_recognition.face_encodings(img_rgb)[0]
            encodeList.append(encode)
            nama_asli = os.path.splitext(cl)[0].split('_')[0]
            classNames.append(nama_asli)
        except IndexError:
            print(f"Wajah tidak terdeteksi dengan jelas pada gambar: {cl}")
    return encodeList, classNames

encodeListKnown, classNames = findEncodings(images, gambar_files)
print(f"Selesai! {len(encodeListKnown)} data wajah dikenali. Memulai kamera...")

###**Bagian 4 — Fungsi Pencatatan Absensi ke File CSV**

1. os.path.isfile() → Memeriksa apakah file CSV sudah tersedia. Apabila belum ada, sistem akan membuat file baru secara otomatis beserta header kolom yang telah ditentukan. Langkah ini memastikan program dapat berjalan dengan baik sejak pertama kali dieksekusi.
2. datetime.now().strftime() → Mengambil tanggal dan waktu saat ini kemudian memformatnya sesuai kebutuhan. Format %Y-%m-%d menghasilkan representasi seperti 2025-01-15, dan %H:%M:%S menghasilkan representasi seperti 08:30:45. Format ini dipilih karena konsisten dan memudahkan proses pengurutan data.
3. df[(df['Nama'] == nama) & (df['Tanggal'] == tanggal_hari_ini)] → Merupakan operasi filtering pandas untuk memeriksa apakah individu dengan nama yang sama telah tercatat pada tanggal yang sama. Pemeriksaan ini sangat penting karena kamera beroperasi secara berkelanjutan dan tanpa mekanisme ini, satu individu berpotensi dicatat ratusan kali dalam satu sesi.
4. if sudah_absen.empty → Data absensi hanya akan ditambahkan apabila hasil filter di atas menghasilkan DataFrame kosong, yang berarti individu tersebut belum tercatat pada hari yang bersangkutan.
5. df.loc[len(df)] → Menambahkan baris data baru di bagian akhir DataFrame.


Penggunaan pandas pada fungsi ini memberikan kemudahan dalam melakukan operasi filtering data yang kompleks hanya dengan satu baris kode. Apabila menggunakan pendekatan file teks biasa, proses pengecekan duplikasi harus dilakukan secara manual baris per baris, yang jauh lebih rentan terhadap kesalahan logika.

In [ ]:
# Fungsi untuk mencatat ke Excel/CSV
def catat_absensi(nama):
    file_excel = 'Absensi.csv'

    # Buat file jika belum ada
    if not os.path.isfile(file_excel):
        df = pd.DataFrame(columns=['Nama', 'Tanggal', 'Waktu Hadir'])
        df.to_csv(file_excel, index=False)

    df = pd.read_csv(file_excel)

    sekarang = datetime.now()
    tanggal_hari_ini = sekarang.strftime('%Y-%m-%d')
    waktu_sekarang = sekarang.strftime('%H:%M:%S')

    # Cek apakah orang ini sudah absen hari ini agar tidak double-input
    sudah_absen = df[(df['Nama'] == nama) & (df['Tanggal'] == tanggal_hari_ini)]

    if sudah_absen.empty:
        df.loc[len(df)] = [nama, tanggal_hari_ini, waktu_sekarang]
        df.to_csv(file_excel, index=False)
        print(f"Berhasil mencatat kehadiran: {nama} pada {waktu_sekarang}")

###**Bagian 5 — Inisialisasi Kamera dan Pengambilan Frame**

1. cv2.VideoCapture(0) → Menginisialisasi kamera pada indeks 0, yang merujuk pada kamera utama yang terpasang pada perangkat. Apabila menggunakan kamera eksternal, nilai indeks dapat diubah menjadi 1 atau lebih.
2. cap.read() → Mengambil satu frame dari aliran video kamera. Fungsi ini mengembalikan dua nilai, yaitu success yang menunjukkan keberhasilan pembacaan frame, dan img yang merupakan data frame tersebut.
3. cv2.resize(..., 0.25, 0.25) → Memperkecil dimensi frame menjadi 25% dari ukuran aslinya sebelum dilakukan proses deteksi wajah. Hal ini merupakan kompromi yang disengaja antara akurasi dan kecepatan pemrosesan.
4. face_recognition.face_locations() → Mendeteksi dan mengembalikan koordinat kotak pembatas wajah yang ditemukan pada frame yang telah diperkecil.
5. face_recognition.face_encodings() → Mengonversi setiap wajah yang terdeteksi menjadi encoding 128 dimensi, sebagaimana yang dilakukan pada tahap pemrosesan foto referensi.


Proses pengenalan wajah membutuhkan sumber daya komputasi yang cukup besar. Apabila pemrosesan dilakukan langsung pada frame beresolusi penuh, kecepatan pemrosesan akan turun secara signifikan hingga hanya menghasilkan 1 hingga 2 frame per detik. Dengan memperkecil frame sebesar 75%, kecepatan pemrosesan dapat meningkat secara drastis sehingga sistem dapat berjalan secara real-time.

In [ ]:
# Membuka Kamera
cap = cv2.VideoCapture(0)

while True:
    success, img = cap.read()
    if not success:
        break

    # Perkecil ukuran frame agar proses deteksi lebih ringan dan real-time
    imgS = cv2.resize(img, (0, 0), None, 0.25, 0.25)
    imgS = cv2.cvtColor(imgS, cv2.COLOR_BGR2RGB)

    # Deteksi lokasi wajah dan encodings pada frame saat ini
    facesCurFrame = face_recognition.face_locations(imgS)
    encodesCurFrame = face_recognition.face_encodings(imgS, facesCurFrame)

###**Bagian 6 — Pencocokan Wajah dan Visualisasi Hasil**

1. face_recognition.compare_faces(..., tolerance=0.5) → Membandingkan encoding wajah yang terdeteksi di kamera dengan seluruh encoding data referensi. Nilai tolerance=0.5 merupakan ambang batas kecocokan — jarak di bawah nilai ini dianggap sebagai kecocokan yang valid. Nilai default library ini adalah 0.6, namun penggunaan nilai 0.5 memberikan tingkat keketatan yang lebih tinggi sehingga meminimalkan risiko false positive.
2. face_recognition.face_distance() → Menghitung jarak numerik antara encoding wajah dari kamera dengan setiap data referensi yang tersimpan. Semakin kecil nilai jarak tersebut, semakin tinggi tingkat kemiripan kedua wajah.
3. np.argmin(faceDis) → Mengembalikan indeks dari nilai jarak terkecil, yang menunjukkan data referensi dengan tingkat kemiripan tertinggi terhadap wajah yang sedang terdeteksi.
4. if matches[matchIndex] → Melakukan verifikasi ganda bahwa wajah yang paling mirip tersebut juga telah melewati ambang batas tolerance yang ditetapkan.
5. y1*4, x2*4, y2*4, x1*4 → Koordinat wajah yang diperoleh berasal dari frame yang telah diperkecil menjadi 25%, sehingga seluruh koordinat harus dikalikan dengan 4 untuk dikembalikan ke skala frame asli sebelum proses visualisasi dilakukan.
6. cv2.rectangle() pertama → Menggambar kotak pembatas berwarna hijau di sekeliling area wajah yang terdeteksi.
7. cv2.rectangle() kedua → Menggambar kotak solid berwarna hijau di bagian bawah kotak wajah sebagai latar belakang teks nama.
8. cv2.putText() → Menuliskan nama individu yang terdeteksi di atas kotak solid tersebut dengan warna putih agar mudah terbaca.


Penggunaan dua metode secara bersamaan, yaitu compare_faces dan face_distance, merupakan pendekatan yang lebih andal dibandingkan menggunakan salah satunya saja. Fungsi compare_faces menentukan apakah sebuah wajah memenuhi ambang batas kecocokan, sedangkan face_distance dengan argmin memastikan bahwa dari semua kandidat yang memenuhi syarat, sistem memilih kandidat dengan tingkat kemiripan tertinggi.

In [ ]:
    for encodeFace, faceLoc in zip(encodesCurFrame, facesCurFrame):
        # Bandingkan wajah di kamera dengan data referensi
        matches = face_recognition.compare_faces(encodeListKnown, encodeFace, tolerance=0.5)
        faceDis = face_recognition.face_distance(encodeListKnown, encodeFace)

        matchIndex = np.argmin(faceDis)

        if matches[matchIndex]:
            nama_terdeteksi = classNames[matchIndex].upper()

            # Kembalikan koordinat bounding box ke ukuran asli
            y1, x2, y2, x1 = faceLoc
            y1, x2, y2, x1 = y1*4, x2*4, y2*4, x1*4

            # Gambar Bounding Box otomatis
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.rectangle(img, (x1, y2-35), (x2, y2), (0, 255, 0), cv2.FILLED)
            cv2.putText(img, nama_terdeteksi, (x1+6, y2-6), cv2.FONT_HERSHEY_COMPLEX, 0.8, (255, 255, 255), 2)

            # Panggil fungsi pencatatan
            catat_absensi(nama_terdeteksi)

###**Bagian 7 — Penampilan Kamera dan Penghentian Program**

1. cv2.imshow() → Menampilkan frame pada sebuah jendela dengan judul yang telah ditentukan. Frame yang ditampilkan adalah frame dengan ukuran asli, bukan yang telah diperkecil, sehingga kualitas tampilan tetap terjaga.
2. cv2.waitKey(1) → Menunggu masukan dari keyboard selama 1 milidetik pada setiap frame. Nilai 1 milidetik ini sangat penting — apabila diisi dengan nilai 0, program akan berhenti dan menunggu masukan keyboard tanpa batas waktu, sehingga tampilan kamera tidak dapat berjalan secara real-time.
3. & 0xFF == 27 → Angka 27 merupakan kode ASCII dari tombol ESC. Operasi bitwise & 0xFF diterapkan untuk memastikan kompatibilitas lintas sistem operasi, mengingat beberapa sistem operasi dapat mengembalikan nilai keyboard yang melebihi 8 bit.
4. cap.release() → Melepaskan sumber daya kamera agar perangkat kamera tidak dalam kondisi terkunci setelah program selesai dieksekusi. Tanpa pemanggilan fungsi ini, kamera kemungkinan tidak dapat digunakan oleh aplikasi lain hingga kernel Python di-restart.
5. cv2.destroyAllWindows() → Menutup seluruh jendela OpenCV yang sedang aktif secara terstruktur dan bersih.


Pemanggilan cap.release() dan cv2.destroyAllWindows() merupakan bagian dari praktik pengelolaan sumber daya yang baik dalam pemrograman. Tanpa kedua fungsi ini, terdapat risiko bahwa jendela kamera akan tetap aktif di latar belakang dan sumber daya kamera tidak dibebaskan dengan semestinya setelah program dihentikan.

In [ ]:
    cv2.imshow('Sistem Absensi Deteksi Wajah', img)

    # Tekan 'ESC' untuk keluar
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

##**Hasil Dari Excel**

File Absensi.csv merupakan output langsung yang dihasilkan oleh fungsi catat_absensi() dalam program. File ini terdiri dari tiga kolom utama, yaitu Nama, Tanggal, dan Waktu Hadir, yang masing-masing menyimpan informasi identitas individu, tanggal kehadiran, dan waktu kehadiran secara tepat.
Berdasarkan isi file tersebut, terdapat beberapa hal yang dapat dijelaskan sebagai berikut.

1. Kolom Nama menyimpan nama individu dalam format huruf kapital seluruhnya. Hal ini sesuai dengan kode classNames[matchIndex].upper() pada program, di mana nama yang diambil dari nama file foto referensi secara otomatis dikonversi menjadi huruf kapital sebelum dicatat ke dalam file CSV.
2. Kolom Tanggal menggunakan format YYYY-MM-DD, yang dalam file ini mencakup rentang tanggal dari 9 April 2026 hingga 11 April 2026. Format ini dipilih dalam program menggunakan strftime('%Y-%m-%d') karena bersifat konsisten dan memudahkan proses pengurutan data secara kronologis.
3. Kolom Waktu Hadir mencatat jam, menit, dan detik pada saat wajah individu berhasil dideteksi dan dikenali oleh sistem, menggunakan format HH:MM:SS.
4. Mekanisme Anti-Duplikasi Berjalan dengan Baik. Dapat diamati bahwa setiap individu hanya tercatat satu kali per hari meskipun kamera beroperasi secara berkelanjutan. Sebagai contoh, FADLAN RIZKY LUBIS tercatat pada tanggal 9 April, 10 April, dan 11 April masing-masing hanya satu kali. Hal ini membuktikan bahwa logika pengecekan sudah_absen.empty pada fungsi catat_absensi() berfungsi sebagaimana mestinya.
5. Total Individu Terdaftar yang berhasil tercatat dalam file ini berjumlah lima orang, yaitu FADLAN RIZKY LUBIS, M.ARLAND HILLERY, FARIZ ARDITIO ARAFAT, dan WAHID DIO SAPUTRA, serta ADITYA RAHMAT. Kelima individu tersebut memiliki foto referensi yang tersimpan di folder foto_wajah dan encoding wajahnya telah berhasil diekstraksi oleh sistem.
6. Konsistensi Pencatatan pada tanggal 10 April 2026 menunjukkan bahwa sistem mampu mendeteksi dan mencatat empat individu yang berbeda dalam satu sesi operasional, yang membuktikan bahwa sistem dapat menangani lebih dari satu individu secara bergantian dalam satu hari.

In [ ]:
Nama,Tanggal,Waktu Hadir
FADLAN RIZKY LUBIS,2026-04-09,21:51:01
FADLAN RIZKY LUBIS,2026-04-10,13:30:35
M.ARLAND HILLERY,2026-04-10,13:34:20
FARIZ ARDITIO ARAFAT,2026-04-10,14:30:13
WAHID DIO SAPUTRA,2026-04-10,17:16:53
FADLAN RIZKY LUBIS,2026-04-11,18:29:11
ADITYA RAHMAT,2026-04-11,21:24:06
M.ARLAND HILLERY,2026-04-11,22:30:08

##**Kesimpulan**

Sistem absensi berbasis deteksi wajah yang dibangun dalam program ini merupakan sebuah implementasi computer vision yang menggabungkan beberapa teknologi secara terpadu. Program ini memanfaatkan OpenCV sebagai antarmuka kamera dan visualisasi, library face_recognition yang berlandaskan pada model deep learning dari dlib sebagai inti pengenalan wajah, serta pandas sebagai sistem pencatatan data yang terstruktur.

Secara keseluruhan, alur kerja sistem ini terbagi menjadi dua tahap utama. Tahap pertama adalah tahap inisialisasi, di mana program membaca seluruh foto referensi dari folder foto_wajah, mengekstraksi encoding wajah dari setiap foto tersebut, dan menyimpannya di dalam memori sebagai data pembanding. Tahap kedua adalah tahap operasional secara real-time, di mana kamera aktif secara berkelanjutan, setiap frame yang masuk diproses untuk mendeteksi keberadaan wajah, encoding wajah yang terdeteksi dibandingkan dengan data referensi menggunakan perhitungan jarak Euclidean, dan apabila kecocokan ditemukan, nama individu ditampilkan pada layar serta dicatat ke dalam file Absensi.csv dengan mekanisme anti-duplikasi per hari.

Efisiensi sistem dijaga melalui teknik pemotongan resolusi frame sebesar 75% sebelum proses deteksi dilakukan, sehingga sistem mampu berjalan secara real-time tanpa membutuhkan perangkat keras yang terlalu tinggi spesifikasinya. Keandalan sistem dijaga melalui penggunaan dua metode pencocokan secara bersamaan, yaitu compare_faces untuk memverifikasi ambang batas kecocokan dan face_distance dengan np.argmin untuk memilih kandidat dengan tingkat kemiripan tertinggi. Hal ini meminimalkan kemungkinan terjadinya false positive dalam proses identifikasi.

Bukti nyata keberhasilan sistem ini dapat dilihat langsung pada file Absensi.csv yang telah dihasilkan, di mana pencatatan kehadiran berlangsung secara akurat, konsisten, dan bebas dari data duplikat selama tiga hari operasional berturut-turut dari tanggal 9 April hingga 11 April 2026.